# Metody Sztucznej Inteligencji w Cyberbezpieczeństwie
# Wykrywanie ataków DDOS - klasyfikacja binarna 

# Dataset: CIC-DDos2019

## Wczytanie zależności

In [11]:
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

## Wczytanie datnych z katalogów i połączenie w jeden dataset
### Osobny dataset dla danych treningowych oraz testowych

In [ ]:
TRAIN_DATASET_PATH = "dataset/01-12"
TEST_DATASET_PATH = "dataset/03-11"

def load_data(folder_path, max_rows_per_file=None):
    dfs = []
    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            file_path = os.path.join(folder_path, file)
            #print(f"Znaleziono plik: {file_path}")
            try:
                temp_df = pd.read_csv(file_path, nrows=max_rows_per_file)
                dfs.append(temp_df)

            except Exception as e:
                print(f"Error loading {file}: {e}")

    df = pd.concat(dfs, ignore_index=True)
    return df

train_df = load_data(TRAIN_DATASET_PATH, max_rows_per_file=50000)
test_df = load_data(TEST_DATASET_PATH, max_rows_per_file=50000)

C:\Users\kradl\AppData\Local\Temp\ipykernel_1360\2865012015.py:11: DtypeWarning: Columns (0: SimillarHTTP) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path, nrows=max_rows_per_file)


(550000, 88)
(350000, 88)


In [ ]:
print(train_df.shape)
train_df.info()
train_df.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 550000 entries, 0 to 549999
Data columns (total 88 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Unnamed: 0                    550000 non-null  int64  
 1   Flow ID                       550000 non-null  str    
 2    Source IP                    550000 non-null  str    
 3    Source Port                  550000 non-null  int64  
 4    Destination IP               550000 non-null  str    
 5    Destination Port             550000 non-null  int64  
 6    Protocol                     550000 non-null  int64  
 7    Timestamp                    550000 non-null  str    
 8    Flow Duration                550000 non-null  int64  
 9    Total Fwd Packets            550000 non-null  int64  
 10   Total Backward Packets       550000 non-null  int64  
 11  Total Length of Fwd Packets   550000 non-null  float64
 12   Total Length of Bwd Packets  550000 non-null  float64


,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,425,172.16.0.5-192.168.50.1-634-60495-17,172.16.0.5,634,192.168.50.1,60495,17,2018-12-01 10:51:39.813448,28415,97,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
1,430,172.16.0.5-192.168.50.1-60495-634-17,192.168.50.1,634,172.16.0.5,60495,17,2018-12-01 10:51:39.820842,2,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,DrDoS_DNS
2,1654,172.16.0.5-192.168.50.1-634-46391-17,172.16.0.5,634,192.168.50.1,46391,17,2018-12-01 10:51:39.852499,48549,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
3,2927,172.16.0.5-192.168.50.1-634-11894-17,172.16.0.5,634,192.168.50.1,11894,17,2018-12-01 10:51:39.890213,48337,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
4,694,172.16.0.5-192.168.50.1-634-27878-17,172.16.0.5,634,192.168.50.1,27878,17,2018-12-01 10:51:39.941151,32026,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS


In [ ]:
print(test_df.shape)
test_df.info()
test_df.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 350000 entries, 0 to 349999
Data columns (total 88 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Unnamed: 0                    350000 non-null  int64  
 1   Flow ID                       350000 non-null  str    
 2    Source IP                    350000 non-null  str    
 3    Source Port                  350000 non-null  int64  
 4    Destination IP               350000 non-null  str    
 5    Destination Port             350000 non-null  int64  
 6    Protocol                     350000 non-null  int64  
 7    Timestamp                    350000 non-null  str    
 8    Flow Duration                350000 non-null  int64  
 9    Total Fwd Packets            350000 non-null  int64  
 10   Total Backward Packets       350000 non-null  int64  
 11  Total Length of Fwd Packets   350000 non-null  float64
 12   Total Length of Bwd Packets  350000 non-null  float64


,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,13605,172.16.0.5-192.168.50.4-870-2908-17,172.16.0.5,870,192.168.50.4,2908,17,2018-11-03 10:09:00.565557,1,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,NetBIOS
1,62631,172.16.0.5-192.168.50.4-871-53796-17,172.16.0.5,871,192.168.50.4,53796,17,2018-11-03 10:09:00.565559,48,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,NetBIOS
2,143869,172.16.0.5-192.168.50.4-648-40660-17,172.16.0.5,648,192.168.50.4,40660,17,2018-11-03 10:09:00.565608,1,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,NetBIOS
3,16171,172.16.0.5-192.168.50.4-872-54308-17,172.16.0.5,872,192.168.50.4,54308,17,2018-11-03 10:09:00.565993,1,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,NetBIOS
4,80845,172.16.0.5-192.168.50.4-873-40653-17,172.16.0.5,873,192.168.50.4,40653,17,2018-11-03 10:09:00.565994,1,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,NetBIOS
